# Quantitative Evaluation

## FID, Precision, Recall, and Conditional Accuracy

This notebook evaluates and compares two conditional image generation models on Fashion-MNIST:
- **Conditional GAN** (with gradient penalty, label smoothing, and instance noise)
- **Conditional Diffusion Model** (DDPM with U-Net)

We compute standard metrics to quantify sample quality, diversity, and conditioning accuracy.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from scipy.linalg import sqrtm
import os
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Constants
BATCH_SIZE = 128
NUM_CLASSES = 10
IMG_SIZE = 28
CHANNELS = 1
NUM_SAMPLES = 10000  # Total samples to generate (1000 per class)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1] for consistency
])

train_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 1. Load Pre-trained Models

In [ ]:
# GAN Architecture copied from training notebook
class Generator(nn.Module):
    """Conditional Generator for Fashion-MNIST"""
    def __init__(self, z_dim=256, num_classes=10, img_size=28, channels=1):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, z_dim)
        self.init_size = img_size // 4  # 7
        
        self.l1 = nn.Sequential(
            nn.Linear(z_dim * 2, 128 * self.init_size ** 2),
            nn.BatchNorm1d(128 * self.init_size ** 2),
            nn.ReLU(inplace=True),
        )
        
        self.conv_blocks = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, channels, 4, stride=2, padding=1),
            nn.Tanh(),
        )
    
    def forward(self, z, labels):
        label_emb = self.label_emb(labels)
        gen_input = torch.cat((z, label_emb), dim=1)
        out = self.l1(gen_input)
        out = out.view(out.size(0), 128, self.init_size, self.init_size)
        return self.conv_blocks(out)


class Discriminator(nn.Module):
    """Conditional Discriminator for Fashion-MNIST"""
    def __init__(self, img_size=28, channels=1, num_classes=10):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, 1)
        self.img_size = img_size
        
        self.model = nn.Sequential(
            nn.Conv2d(channels + 1, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(128 * (img_size // 4) * (img_size // 4), 1),
        )
    
    def forward(self, img, labels):
        label_input = self.label_emb(labels).view(-1, 1, 1, 1)
        label_input = label_input.expand(-1, 1, self.img_size, self.img_size)
        x = torch.cat((img, label_input), dim=1)
        return self.model(x)


# Load GAN models
z_dim = 256
G = Generator(z_dim, NUM_CLASSES, IMG_SIZE, CHANNELS).to(device)
D = Discriminator(IMG_SIZE, CHANNELS, NUM_CLASSES).to(device)

# Load weights
G.load_state_dict(torch.load('models/gan_generator_best.pth', map_location=device))
D.load_state_dict(torch.load('models/gan_discriminator_best.pth', map_location=device))

G.eval()
D.eval()
print("GAN models loaded successfully!")

In [ ]:
диффузионная модель дописать

## 2. Generate Samples

We generate **10,000 images** from each model (1,000 per class) for fair comparison.

- **GAN**: Direct generation, outputs 28×28 in range [-1,1], convert to [0,1]
- **Diffusion**: Generates 32×32 images, resize to 28×28 for consistent evaluation

In [ ]:
def generate_gan_samples(generator, num_samples=10000, batch_size=128):
    """Generate samples from GAN (28×28, output range [0,1])"""
    generator.eval()
    samples = []
    labels_list = []
    
    num_per_class = num_samples // NUM_CLASSES
    batches_per_class = num_per_class // batch_size
    
    with torch.no_grad():
        for label in range(NUM_CLASSES):
            for _ in tqdm(range(batches_per_class), desc=f"GAN Class {class_names[label]}"):
                z = torch.randn(batch_size, z_dim, device=device)
                labels = torch.full((batch_size,), label, device=device)
                imgs = generator(z, labels)  # Range [-1, 1]
                imgs = (imgs + 1) / 2  # Convert to [0, 1]
                samples.append(imgs.cpu())
                labels_list.append(labels.cpu())
    
    # Handle remainder
    remaining = num_per_class - (batches_per_class * batch_size)
    if remaining > 0:
        z = torch.randn(remaining, z_dim, device=device)
        labels = torch.full((remaining,), NUM_CLASSES - 1, device=device)
        imgs = generator(z, labels)
        imgs = (imgs + 1) / 2
        samples.append(imgs.cpu())
        labels_list.append(labels.cpu())
    
    return torch.cat(samples), torch.cat(labels_list)

print("Generating GAN samples...")
gan_samples, gan_labels = generate_gan_samples(G, NUM_SAMPLES)
print(f"GAN samples shape: {gan_samples.shape}")
print(f"GAN labels shape: {gan_labels.shape}")

In [ ]:
def generate_diffusion_samples(diffusion, model, num_samples=10000, batch_size=128):
    дописать

print("Generating Diffusion samples...")
diffusion_samples, diffusion_labels = generate_diffusion_samples(diffusion, model, NUM_SAMPLES)
print(f"Diffusion samples shape: {diffusion_samples.shape}")
print(f"Diffusion labels shape: {diffusion_labels.shape}")

In [ ]:
def visualize_samples(samples, labels, title, nrow=10):
    """Display a grid of generated samples"""
    samples = samples[:20]  # Show first 20 samples
    grid = make_grid(samples, nrow=nrow, normalize=False, pad_value=1)
    
    plt.figure(figsize=(15, 4))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.title(title, fontsize=14)
    plt.axis('off')
    
    # Add class labels
    for i in range(10):
        plt.text(i * 28 + 14, -5, class_names[labels[i].item()], 
                 ha='center', va='bottom', fontsize=8, rotation=45)
    plt.tight_layout()
    plt.show()

visualize_samples(gan_samples, gan_labels, "GAN - Generated Samples (10 classes, 2 per class)")
visualize_samples(diffusion_samples, diffusion_labels, "Diffusion - Generated Samples (10 classes, 2 per class)")

## 3. Feature Extractor for Metrics

We train a simple CNN classifier on real Fashion-MNIST data to extract features for FID, Precision, and Recall computation.

In [ ]:
class FeatureExtractor(nn.Module):
    """CNN for feature extraction and classification on Fashion-MNIST"""
    def __init__(self, num_classes=10, feature_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        self.classifier = nn.Linear(128, num_classes)
    
    def forward(self, x, return_features=False):
        features = self.features(x)
        if return_features:
            return features
        return self.classifier(features)

# Train feature extractor on real data
def train_feature_extractor(model, train_loader, epochs=10, lr=0.001):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        epoch_loss = 0
        correct = 0
        total = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            pbar.set_postfix(Loss=loss.item(), Acc=100.*correct/total)
    
    return model

# Create feature extractor
feature_extractor = FeatureExtractor().to(device)
print("Training feature extractor on real Fashion-MNIST...")
feature_extractor = train_feature_extractor(feature_extractor, train_loader, epochs=10)

# Evaluate on test set
feature_extractor.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = feature_extractor(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

print(f"Feature extractor test accuracy: {100.*correct/total:.2f}%")

## 4. Compute Evaluation Metrics

### 4.1 FID (Fréchet Inception Distance)

FID measures the distance between real and generated feature distributions. Lower is better.

In [ ]:
def compute_fid(real_features, fake_features):
    """Compute FID between real and fake feature distributions"""
    mu1 = np.mean(real_features, axis=0)
    mu2 = np.mean(fake_features, axis=0)
    sigma1 = np.cov(real_features, rowvar=False)
    sigma2 = np.cov(fake_features, rowvar=False)
    
    # Calculate sqrt of matrix product
    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    fid = np.sum((mu1 - mu2)**2) + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fid

# Extract features from real data
def extract_features(model, dataloader, num_samples=None):
    """Extract features from images using the trained feature extractor"""
    model.eval()
    features = []
    labels_list = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting features"):
            images = images.to(device)
            feat = model(images, return_features=True)
            features.append(feat.cpu().numpy())
            labels_list.append(labels.numpy())
            
            if num_samples and len(features) * BATCH_SIZE >= num_samples:
                break
    
    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    
    if num_samples:
        features = features[:num_samples]
        labels = labels[:num_samples]
    
    return features, labels

# Extract features from real test data
real_features, real_labels = extract_features(feature_extractor, test_loader, NUM_SAMPLES)

# Extract features from GAN samples
gan_dataloader = DataLoader(TensorDataset(gan_samples, gan_labels), batch_size=BATCH_SIZE)
gan_features, gan_labels = extract_features(feature_extractor, gan_dataloader, NUM_SAMPLES)

# Extract features from Diffusion samples
diffusion_dataloader = DataLoader(TensorDataset(diffusion_samples, diffusion_labels), batch_size=BATCH_SIZE)
diffusion_features, diffusion_labels_extracted = extract_features(feature_extractor, diffusion_dataloader, NUM_SAMPLES)

# Compute FID
fid_gan = compute_fid(real_features, gan_features)
fid_diffusion = compute_fid(real_features, diffusion_features)

print(f"GAN FID: {fid_gan:.4f}")
print(f"Diffusion FID: {fid_diffusion:.4f}")

### 4.2 Precision and Recall

Precision measures sample quality (fraction of generated samples near real manifold).  
Recall measures diversity (fraction of real samples covered by generated distribution).

In [ ]:
def compute_precision_recall(real_features, fake_features, k=3):
    """Compute precision and recall in feature space"""
    real_features = torch.tensor(real_features).float()
    fake_features = torch.tensor(fake_features).float()
    
    # Compute pairwise distances
    dist_real_to_fake = torch.cdist(real_features, fake_features)
    dist_fake_to_real = torch.cdist(fake_features, real_features)
    
    # For each real sample, find k-th nearest neighbor distance
    real_knn_dist = torch.topk(dist_real_to_fake, k, largest=False)[0][:, -1]
    threshold = torch.median(real_knn_dist)
    
    # Precision: fake samples within threshold of any real sample
    nearest_dist_to_real = dist_fake_to_real.min(dim=1)[0]
    precision = (nearest_dist_to_real < threshold).float().mean().item()
    
    # Recall: real samples within threshold of any fake sample
    nearest_dist_to_fake = dist_real_to_fake.min(dim=1)[0]
    recall = (nearest_dist_to_fake < threshold).float().mean().item()
    
    return precision, recall

precision_gan, recall_gan = compute_precision_recall(real_features, gan_features)
precision_diffusion, recall_diffusion = compute_precision_recall(real_features, diffusion_features)

print(f"GAN - Precision: {precision_gan:.4f}, Recall: {recall_gan:.4f}")
print(f"Diffusion - Precision: {precision_diffusion:.4f}, Recall: {recall_diffusion:.4f}")

### 4.3 Conditional Classification Accuracy

This metric measures how well generated samples match their intended class labels.

In [ ]:
def compute_conditional_accuracy(classifier, samples, labels):
    """Check if generated samples match their intended classes"""
    classifier.eval()
    correct = 0
    total = len(samples)
    per_class_correct = [0] * NUM_CLASSES
    per_class_total = [0] * NUM_CLASSES
    
    with torch.no_grad():
        for i in range(0, total, BATCH_SIZE):
            batch_samples = samples[i:i+BATCH_SIZE].to(device)
            batch_labels = labels[i:i+BATCH_SIZE].to(device)
            
            outputs = classifier(batch_samples)
            predictions = outputs.argmax(dim=1)
            
            for j in range(len(batch_labels)):
                true_label = batch_labels[j].item()
                per_class_total[true_label] += 1
                if predictions[j] == batch_labels[j]:
                    correct += 1
                    per_class_correct[true_label] += 1
    
    accuracy = correct / total
    per_class_accuracy = [per_class_correct[i] / per_class_total[i] if per_class_total[i] > 0 else 0 
                          for i in range(NUM_CLASSES)]
    
    return accuracy, per_class_accuracy

gan_acc, gan_per_class = compute_conditional_accuracy(feature_extractor, gan_samples, gan_labels)
diffusion_acc, diffusion_per_class = compute_conditional_accuracy(feature_extractor, diffusion_samples, diffusion_labels)

print(f"GAN Conditional Accuracy: {gan_acc:.4f} ({gan_acc*100:.2f}%)")
print(f"Diffusion Conditional Accuracy: {diffusion_acc:.4f} ({diffusion_acc*100:.2f}%)")

## 5. Results Summary

In [ ]:
results = {
    'Metric': ['FID ↓', 'Precision ↑', 'Recall ↑', 'Conditional Accuracy ↑'],
    'GAN': [f'{fid_gan:.2f}', f'{precision_gan:.4f}', f'{recall_gan:.4f}', f'{gan_acc:.4f}'],
    'Diffusion': [f'{fid_diffusion:.2f}', f'{precision_diffusion:.4f}', f'{recall_diffusion:.4f}', f'{diffusion_acc:.4f}']
}

# Display results
fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')
table = ax.table(cellText=list(zip(results['Metric'], results['GAN'], results['Diffusion'])),
                 colLabels=['Metric', 'GAN', 'Diffusion'],
                 loc='center',
                 cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.5)
plt.title("Quantitative Comparison: GAN vs Diffusion", fontsize=14, pad=20)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(NUM_CLASSES)
width = 0.35

bars1 = ax.bar(x - width/2, [acc * 100 for acc in gan_per_class], width, label='GAN', alpha=0.8)
bars2 = ax.bar(x + width/2, [acc * 100 for acc in diffusion_per_class], width, label='Diffusion', alpha=0.8)

ax.set_xlabel('Class')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Conditional Accuracy')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()